In [ ]:
!pip install torchmetrics[image]
!pip install --upgrade torchao


import os
import json
import multiprocessing
import shutil
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, List, Optional, Union


import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import torch
import torch.nn.functional as F
from accelerate import Accelerator
from diffusers import AutoencoderKL, DDPMScheduler, StableDiffusionPipeline, UNet2DConditionModel
from peft import LoraConfig, get_peft_model
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader, Dataset
from torchmetrics.image.fid import FrechetInceptionDistance
from torchvision import transforms
from tqdm.auto import tqdm
from transformers import CLIPModel, CLIPProcessor, CLIPTextModel, CLIPTokenizer, PreTrainedTokenizer


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.6/85.6 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 38.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 50.8 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


Unable to import `torchao` Tensor objects. This may affect loading checkpoints serialized with `torchao`
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


Kaggle credentials set.
Kaggle credentials successfully validated.


In [ ]:
import zipfile
import os
from google.colab import drive

drive.mount('/content/drive', force_remount=True)

zip_path = '/content/drive/MyDrive/DL_FinalTerm/lora_nhatbinh_backup.zip'
extract_path = '/content/lora_nhatbinh_weights'

if os.path.exists(zip_path):
    print(f'Zip file found. Starting extraction...')
    try:
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(extract_path)
        print(f'Successfully extracted to: {extract_path}')
    except OSError as e:
        print(f'Extraction failed due to connection issues: {e}')
else:
    print(f'Error: {zip_path} not found. Please ensure the file exists in your Drive.')

Mounted at /content/drive
Zip file found. Starting extraction...
Extraction failed due to connection issues: [Errno 107] Transport endpoint is not connected


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

thedestoroyah_ao_nhat_binh_v3_path = kagglehub.dataset_download('thedestoroyah/ao-nhat-binh-v3')

print('Data source import complete.')


100%|██████████| 39.6M/39.6M [00:00<00:00, 90.9MB/s]

Extracting files...


Data source import complete.


In [ ]:
import torch
import gradio as gr
from diffusers import StableDiffusionPipeline, DPMSolverMultistepScheduler
from peft import PeftModel
from typing import Tuple
from PIL import Image

class NhatBinhInference:
    """
    Lớp quản lý luồng suy luận (Inference Pipeline).
    Sử dụng kỹ thuật Merge and Unload của PEFT để gộp vĩnh viễn trọng số LoRA
    vào mô hình gốc, đảm bảo tính chính xác và tăng tốc độ suy luận.
    """

    def __init__(self, base_model_id: str, lora_weights_path: str, device: str = "cuda"):
        """
        Khởi tạo đối tượng NhatBinhInference.

        Input:
            base_model_id (str): Định danh của mô hình gốc (VD: runwayml/stable-diffusion-v1-5).
            lora_weights_path (str): Đường dẫn đến thư mục chứa file LoRA của PEFT.
            device (str): Thiết bị tính toán ("cuda" để tận dụng GPU).
        """
        self.device = device
        self.base_model_id = base_model_id
        self.lora_weights_path = lora_weights_path

        print("Loading base Stable Diffusion pipeline...")
        self.pipeline = StableDiffusionPipeline.from_pretrained(
            self.base_model_id,
            torch_dtype=torch.float16,
            safety_checker=None
        )

        print("Optimizing scheduler for fast inference...")
        self.pipeline.scheduler = DPMSolverMultistepScheduler.from_config(
            self.pipeline.scheduler.config,
            use_karras_sigmas=True
        )

        print(f"Injecting and merging PEFT LoRA weights from {self.lora_weights_path}...")
        # 1. Bọc U-Net gốc bằng PeftModel để load chính xác định dạng adapter
        self.pipeline.unet = PeftModel.from_pretrained(
            self.pipeline.unet,
            self.lora_weights_path
        )

        # 2. Gộp ma trận LoRA vào ma trận U-Net và giải phóng bộ nhớ của vỏ bọc PEFT
        self.pipeline.unet = self.pipeline.unet.merge_and_unload()

        # Đảm bảo U-Net vẫn ở định dạng fp16 sau khi merge
        self.pipeline.unet.to(dtype=torch.float16)

        self.pipeline.to(self.device)
        print("Inference engine is ready with merged weights.")

    def generate_image(self, prompt: str, negative_prompt: str, num_inference_steps: int, guidance_scale: float, seed: int) -> Image.Image:
        """
        Thực thi quá trình sinh ảnh dựa trên các tham số đầu vào.

        Input:
            prompt (str): Câu lệnh miêu tả nội dung bức ảnh cần sinh.
            negative_prompt (str): Những chi tiết không mong muốn xuất hiện trong ảnh.
            num_inference_steps (int): Số bước khử nhiễu.
            guidance_scale (float): Mức độ tuân thủ prompt.
            seed (int): Hạt giống ngẫu nhiên (-1 để tạo ảnh ngẫu nhiên).

        Output:
            Image.Image: Bức ảnh kết quả định dạng PIL.
        """
        generator = None
        if seed != -1:
            generator = torch.Generator(device=self.device).manual_seed(seed)

        # Tiền xử lý prompt: Ép luôn trigger word vào đầu câu
        trigger_word = "nhatbinh dress, traditional vietnamese clothing, "

        # Nếu người dùng đã tự gõ chữ nhatbinh dress thì không cộng thêm để tránh lặp từ
        if "nhatbinh" not in prompt.lower():
            full_prompt = trigger_word + prompt
        else:
            full_prompt = prompt

        print(f"Generating image for prompt: {full_prompt}")

        with torch.no_grad():
            image = self.pipeline(
                prompt=full_prompt,
                negative_prompt=negative_prompt,
                num_inference_steps=num_inference_steps,
                guidance_scale=guidance_scale,
                generator=generator
            ).images[0]

        return image




In [ ]:
class GradioWebInterface:
    """
    Lớp quản lý giao diện người dùng (UI) bằng thư viện Gradio.
    Thiết kế theo dạng Blocks để tối ưu không gian hiển thị cho các tham số Deep Learning.
    """

    def __init__(self, inference_engine: NhatBinhInference):
        """
        Khởi tạo đối tượng GradioWebInterface.

        Input:
            inference_engine (NhatBinhInference): Đối tượng chứa logic sinh ảnh.
        """
        self.engine = inference_engine
        self.app = self._build_interface()

    def _build_interface(self) -> gr.Blocks:
        """
        Kiến trúc layout HTML/CSS thông qua Gradio Blocks.
        """
        # Khởi tạo khối giao diện với giao diện sáng/tối tự động
        with gr.Blocks(title="Nhat Binh Dress Generator", theme=gr.themes.Soft()) as interface:
            gr.Markdown("#Stable Diffusion: Nhat Binh Dress Generator")
            gr.Markdown("Hệ thống sinh ảnh Cổ phục Việt Nam (Áo Nhật Bình) sử dụng kỹ thuật tinh chỉnh Low-Rank Adaptation (LoRA) trên kiến trúc Stable Diffusion v1.5.")

            with gr.Row():
                # Cột bên trái: Các tham số đầu vào
                with gr.Column(scale=1):
                    prompt = gr.Textbox(
                        label="Câu lệnh (Prompt)",
                        placeholder="a beautiful woman standing in an ancient palace, highly detailed...",
                        lines=3
                    )
                    negative_prompt = gr.Textbox(
                        label="Câu lệnh loại trừ (Negative Prompt)",
                        value="ugly, blurry, bad anatomy, bad hands, missing fingers, deformed, low quality, mutation",
                        lines=2
                    )

                    with gr.Row():
                        steps = gr.Slider(label="Inference Steps", minimum=10, maximum=50, value=25, step=1)
                        guidance = gr.Slider(label="Guidance Scale (CFG)", minimum=1.0, maximum=15.0, value=7.5, step=0.5)

                    seed = gr.Number(label="Seed (-1 cho ngẫu nhiên)", value=-1, precision=0)

                    generate_btn = gr.Button("Sinh Ảnh (Generate)", variant="primary")

                # Cột bên phải: Nơi hiển thị kết quả
                with gr.Column(scale=1):
                    output_image = gr.Image(label="Kết quả (Output)", type="pil")

            # Khung chứa các ví dụ (Examples) để click vào là chạy thử ngay
            gr.Examples(
                examples=[
                    ["portrait of a royal princess, looking at viewer, cinematic lighting, masterpiece", "blurry, low quality", 25, 7.5, 42],
                    ["a young girl holding a lotus flower, standing near a lake, highly detailed face", "ugly, deformed, bad hands", 25, 7.0, 1024]
                ],
                inputs=[prompt, negative_prompt, steps, guidance, seed],
                outputs=output_image,
                fn=self.engine.generate_image,
                cache_examples=False
            )

            # Gắn sự kiện click nút bấm vào hàm sinh ảnh của Engine
            generate_btn.click(
                fn=self.engine.generate_image,
                inputs=[prompt, negative_prompt, steps, guidance, seed],
                outputs=output_image
            )

        return interface

    def launch(self, share: bool = True) -> None:
        """
        Khởi chạy máy chủ Web cục bộ và tạo Public URL.

        Input:
            share (bool): Nếu True, Gradio sẽ tạo một đường dẫn public (.gradio.live)
                          có giá trị trong 72 giờ để truy cập từ xa.
        """
        print("Launching Gradio Web Server...")
        self.app.launch(share=share, inline=False)

In [ ]:
def main_inference_demo() -> None:
    """
    Hàm thực thi chính cho quá trình chạy Web Demo.
    Khởi tạo mô hình và giao diện người dùng, sau đó bật server.
    """
    print("Initializing Web Demo System...")

    base_model_path = "runwayml/stable-diffusion-v1-5"
    lora_path = "/content/lora_nhatbinh_weights"

    # 1. Khởi tạo Engine
    inference_engine = NhatBinhInference(
        base_model_id=base_model_path,
        lora_weights_path=lora_path
    )

    # 2. Khởi tạo và chạy UI
    web_app = GradioWebInterface(inference_engine=inference_engine)
    web_app.launch(share=True)



Initializing Web Demo System...
Loading base Stable Diffusion pipeline...


Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
You have disabled the safety checker for <class 'diffusers.pipelines.stable_diffusion.pipeline_stable_diffusion.StableDiffusionPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all public facing circumstances, disabling it only for use-cases that involve analyzing network behavior or auditing its resul

Optimizing scheduler for fast inference...
Injecting and merging PEFT LoRA weights from /content/lora_nhatbinh_weights...
Inference engine is ready with merged weights.


/tmp/ipykernel_2232/3049307479.py:119: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="Nhat Binh Dress Generator", theme=gr.themes.Soft()) as interface:


Launching Gradio Web Server...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f8edc75c3aafc0ca05.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
if __name__ == "__main__":
    main_inference_demo()